# 06 Estimation sample

Size filter (€25 million), median imputation (imputed sample only), sample filter, winsorisation at the 1st and 99th percentiles and rank transformation onto (−1, +1), each applied within the month. Produces the complete-case and the imputed samples, saved with the suffix `_mv25`; with `MIN_MV = None` the same steps give the unrestricted samples used as an extension. Section 3.4.6.

## Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd

IN_DIR      = "/content/drive/MyDrive/Thesis/data"   # contains panel_characteristics.parquet
OUT_DIR     = "/content/drive/MyDrive/Thesis/data"   # estimation samples are written here
MASTER_PATH = "/content/drive/MyDrive/Thesis/data/master_universe.csv"

SAMPLE_START = "2000-01-01"
MIN_MV = 25         # 25: main samples, as Drobetz and Otto (2021). None: unrestricted extension
MIN_FIRMS = 50
SEED = 0               # random tie-breaking in the rank transformation

panel = pd.read_parquet(os.path.join(IN_DIR, "panel_characteristics.parquet"))
panel["date"] = pd.to_datetime(panel["date"])
panel = panel.set_index(["symbol", "date"]).sort_index()
print(f"Panel: {len(panel):,} security-months, "
      f"{panel.index.get_level_values('symbol').nunique():,} securities")

SUFFIX = "" if not MIN_MV else f"_mv{int(MIN_MV)}"
print(f"Size filter: {'not applied' if not MIN_MV else f'EUR {MIN_MV:.0f} mn'}  |  file suffix: '{SUFFIX}'")

## Functions

In [ ]:
import numpy as np
import pandas as pd

CHARS = ["size", "bm", "operatingprofitability", "totalassetgrowth", "ret_2_12",
         "issues_1_36", "accrualschange", "roa", "investment", "ret_1",
         "ret_12_36", "dy", "beta", "vola", "turnover", "debttoprice",
         "salestoprice", "cftoprice", "earningstoprice", "issues_1_12",
         "fcdispersion", "grossprofitability"]


def apply_size_filter(p, min_mv=25.0):
    """Drop security-months with market value below min_mv (EUR million) or without a market value."""
    keep = p["MV"] >= min_mv
    return p[keep].copy(), p[~keep]


def impute_cross_sectional_median(p, chars=CHARS):
    """Replace missing characteristics with the cross-sectional median of the month; rows with no characteristic are left untouched."""
    p = p.copy()
    has_any = p[chars].notna().any(axis=1)
    for c in chars:
        med = p.groupby(level="date")[c].transform("median")
        p.loc[has_any, c] = p.loc[has_any, c].fillna(med[has_any])
    return p


def filter_sample(p, chars=CHARS, min_firms=50, breadth_mv=25.0):
    """Keep rows with all characteristics and a target, in months with at least min_firms such firms above breadth_mv."""
    ok = p[chars + ["target"]].notna().all(axis=1)
    d = p[ok].copy()
    n = (d["MV"] >= breadth_mv).groupby(level="date").sum()
    valid = n[n >= min_firms].index
    dropped = sorted(set(d.index.get_level_values("date")) - set(valid))
    d = d[d.index.get_level_values("date").isin(valid)]
    return d, dropped


def winsorise(p, cols, lower=0.01, upper=0.99):
    """Cross-sectional winsorisation at the 1st and 99th percentiles, month by month."""
    p = p.copy()
    for c in cols:
        g = p.groupby(level="date")[c]
        lo = g.transform(lambda s: s.quantile(lower))
        hi = g.transform(lambda s: s.quantile(upper))
        p[c] = p[c].clip(lo, hi)
    return p


def rank_transform(p, chars=CHARS, missing=None, seed=0):
    """Cross-sectional rank onto (-1, +1), mapped as 2(r - 0.5)/n - 1.
         Ties are broken at random (seeded), so that a tied rank cannot identify the month;
         imputed values, given by `missing`, are set to zero."""
    rng = np.random.default_rng(seed)
    p = p.copy()
    dates = p.index.get_level_values("date")
    for c in chars:
        obs = p[c].notna().to_numpy() if missing is None else ~missing[c].to_numpy()
        key = pd.DataFrame({"d": dates[obs], "v": p[c].to_numpy()[obs],
                            "u": rng.random(int(obs.sum()))})
        order = key.sort_values(["d", "v", "u"])
        r = order.groupby("d").cumcount().to_numpy() + 1
        n = order.groupby("d")["d"].transform("size").to_numpy()
        ranked = np.empty(len(key))
        ranked[order.index.to_numpy()] = 2.0 * (r - 0.5) / n - 1.0
        out = np.zeros(len(p))
        out[obs] = ranked
        p[c] = out
    return p


# Drobetz and Otto (2021), Table 1: time-series average of the monthly
# cross-sectional mean, and time-series standard deviation of that mean.
DO_MEAN = {"target": 0.0051, "size": 13.22, "bm": -0.55, "operatingprofitability": 0.06,
           "totalassetgrowth": 0.07, "ret_2_12": 0.09, "issues_1_36": 0.09,
           "accrualschange": -0.04, "roa": 0.05, "investment": 1.02, "ret_1": 0.01,
           "ret_12_36": 0.09, "dy": 0.03, "beta": 0.83, "vola": 0.32, "turnover": 0.02,
           "debttoprice": 0.84, "salestoprice": 2.16, "cftoprice": 0.14,
           "earningstoprice": 0.04, "issues_1_12": 0.02, "fcdispersion": -2.05,
           "grossprofitability": 0.30}
DO_STD = {"target": 0.0455, "size": 0.48, "bm": 0.18, "operatingprofitability": 0.01,
          "totalassetgrowth": 0.04, "ret_2_12": 0.20, "issues_1_36": 0.02,
          "accrualschange": 0.01, "roa": 0.01, "investment": 0.08, "ret_1": 0.05,
          "ret_12_36": 0.31, "dy": 0.01, "beta": 0.10, "vola": 0.06, "turnover": 0.01,
          "debttoprice": 0.32, "salestoprice": 0.60, "cftoprice": 0.04,
          "earningstoprice": 0.02, "issues_1_12": 0.01, "fcdispersion": 0.18,
          "grossprofitability": 0.03}


def compare_with_source(d, size_offset=np.log(1000)):
    """Time-series mean and standard deviation of the monthly cross-sectional mean."""
    d = d.copy()
    d["size"] = d["size"] + size_offset
    rows = []
    for c in DO_MEAN:
        m = d.groupby(level="date")[c].mean()
        rows.append({"variable": c, "mean": m.mean(), "DO_mean": DO_MEAN[c],
                     "std": m.std(), "DO_std": DO_STD[c]})
    return pd.DataFrame(rows).set_index("variable")


def latex_comparison_table(cmp, n_obs, firms_per_month):
    label = {c: "Excess return" if c == "target" else "\\texttt{" + c.replace("_", "\\_") + "}"
             for c in cmp.index}
    f = lambda v: f"$-{abs(v):.3f}$" if v < 0 else f"{v:.3f}"
    out = [r"\begin{table}[htbp]", r"\centering\small",
           r"\caption{The estimation sample against Drobetz and Otto (2021)}",
           r"\label{tab:final}", r"\begin{tabular}{@{}lcccc@{}}", r"\toprule",
           r" & \multicolumn{2}{c}{Mean} & \multicolumn{2}{c}{Std.\ dev.} \\",
           r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}",
           r" & This sample & Drobetz--Otto & This sample & Drobetz--Otto \\", r"\midrule"]
    for c, r in cmp.iterrows():
        out.append(f"{label[c]} & {f(r['mean'])} & {f(r['DO_mean'])} & "
                   f"{f(r['std'])} & {f(r['DO_std'])} \\\\")
        if c == "target":
            out.append(r"\addlinespace")
    out += [r"\bottomrule", r"\end{tabular}", r"\\[0.5em]",
            r"\begin{minipage}{0.94\textwidth}",
            r"\scriptsize \textit{Notes.} For each characteristic, the cross-sectional mean is "
            r"computed in every month and the table reports the time-series average of that series "
            r"(\emph{Mean}) together with its standard deviation (\emph{Std.\ dev.}), which measures "
            r"the month-to-month variation of the average firm, not the dispersion across firms "
            r"within a month. Statistics for this sample are computed on the winsorised "
            f"complete-case sample of {n_obs:,}".replace(",", "{,}") +
            r" security-months, before the rank transformation; \texttt{size} is expressed in the "
            r"units of the source by adding $\log 1000$. Figures for Drobetz and Otto (2021) are "
            r"taken from their Table~1 and refer to 832 firms per month across nineteen euro-area "
            r"countries over 1990--2020, against " + f"{firms_per_month:.0f}" +
            r" firms per month across four countries over 2000--2025 here. The excess return is "
            r"monthly and expressed as a decimal.", r"\end{minipage}", r"\end{table}"]
    return "\n".join(out)


## Sample period

In [ ]:
panel = panel[panel.index.get_level_values("date") >= SAMPLE_START]
print(f"From {SAMPLE_START[:7]}: {len(panel):,} security-months, "
      f"{panel.index.get_level_values('symbol').nunique():,} securities")

## Size filter

In [ ]:
if MIN_MV:
    n0 = len(panel)
    cc_before, _ = filter_sample(panel, min_firms=MIN_FIRMS)
    panel, removed = apply_size_filter(panel, MIN_MV)

    print(f"Security-months removed: {len(removed):,} of {n0:,} ({100*len(removed)/n0:.1f}%)")
    print(f"  of which without a market value: {int(removed['MV'].isna().sum()):,}")
    print(f"  median market value of those removed: {removed['MV'].median():.1f} mn")
    print(f"Remaining: {len(panel):,} security-months, "
          f"{panel.index.get_level_values('symbol').nunique():,} securities")

    cc_after, _ = filter_sample(panel, min_firms=MIN_FIRMS)
    print(f"\nEffect on the complete-case sample: {len(cc_before):,} -> {len(cc_after):,} "
          f"({len(cc_before)-len(cc_after):,} removed, "
          f"{100*(len(cc_before)-len(cc_after))/len(cc_before):.1f}%)")
else:
    below = (panel["MV"] < 25).sum()
    print("Size filter not applied (unrestricted extension samples).")
    print(f"For reference, security-months below EUR 25 mn: {below:,} "
          f"({100*below/len(panel):.1f}%)")

In [ ]:
print("Missingness by characteristic, after the size filter:")
miss = (100 * panel[CHARS].isna().mean()).sort_values(ascending=False)
for ch, m in miss.items():
    print(f"  {ch:24s} {m:>5.1f}%")

## Complete-case sample

In [ ]:
cc, dropped_cc = filter_sample(panel, min_firms=MIN_FIRMS)
per_month_cc = cc.groupby(level="date").size()

print(f"Complete case: {len(cc):,} security-months, "
      f"{cc.index.get_level_values('symbol').nunique():,} securities")
print(f"Months failing the breadth requirement: {len(dropped_cc)}")
print(f"Firms per month: mean {per_month_cc.mean():.0f}, "
      f"min {per_month_cc.min():.0f}, max {per_month_cc.max():.0f}")
print(f"Period: {cc.index.get_level_values('date').min():%Y-%m} to "
      f"{cc.index.get_level_values('date').max():%Y-%m}, "
      f"{cc.index.get_level_values('date').nunique()} months")

## Imputed sample

In [ ]:
missing_mask = panel[CHARS].isna()          # which values will be imputed
n_imputed = missing_mask.sum(axis=1).rename("n_imputed")

panel_imp = impute_cross_sectional_median(panel)
im, dropped_im = filter_sample(panel_imp, min_firms=MIN_FIRMS)
im = im.join(n_imputed)
per_month_im = im.groupby(level="date").size()

print(f"Imputed: {len(im):,} security-months, "
      f"{im.index.get_level_values('symbol').nunique():,} securities")
print(f"Firms per month: mean {per_month_im.mean():.0f}, "
      f"min {per_month_im.min():.0f}, max {per_month_im.max():.0f}")
print(f"Relative to complete case: {len(im)/len(cc):.2f}x observations, "
      f"{im.index.get_level_values('symbol').nunique()/cc.index.get_level_values('symbol').nunique():.2f}x securities")

print("\nCharacteristics imputed per observation:")
print(im["n_imputed"].value_counts().sort_index().to_string())
le3, gt10 = (im["n_imputed"] <= 3).sum(), (im["n_imputed"] > 10).sum()
print(f"\nAt most three imputed : {le3:,} ({le3/len(cc):.2f}x the complete-case sample)")
print(f"More than ten imputed : {gt10:,} ({100*gt10/len(im):.1f}%)")

### What the complete-case restriction removes
Medians of beta, volatility and turnover for the recovered set include imputed values; market value is never imputed.

In [ ]:
only_imp = im.index.difference(cc.index)
a = im.loc[only_imp]
print(f"Observations recovered by imputation: {len(only_imp):,}")
print(f"Securities in the recovered set    : {only_imp.get_level_values('symbol').nunique():,}\n")
print(f"{'median of':24s} {'recovered':>12s} {'complete case':>14s}")
for lbl, col in [("market value (mn)", "MV"), ("beta", "beta"),
                 ("volatility", "vola"), ("turnover", "turnover")]:
    print(f"  {lbl:22s} {a[col].median():>12.3f} {cc[col].median():>14.3f}")
if not MIN_MV:
    no_mv = im["MV"].isna()
    print(f"\nImputed sample without an observed market value: {int(no_mv.sum()):,} "
          f"({100*no_mv.mean():.1f}%); their size is the cross-sectional median, and they")
    print("carry no weight in value-weighted portfolios.")

## Winsorisation

In [ ]:
TO_WINSORISE = CHARS + ["target"]
cc_w = winsorise(cc, TO_WINSORISE)
im_w = winsorise(im, TO_WINSORISE)

print("Complete-case sample, effect on dispersion:")
print(f"{'':24s} {'before':>10s} {'after':>10s}")
for ch in ["target", "salestoprice", "debttoprice", "turnover", "beta"]:
    print(f"  {ch:22s} {cc[ch].std():>10.3f} {cc_w[ch].std():>10.3f}")

## Comparison with Drobetz and Otto

In [ ]:
cmp = compare_with_source(cc_w)
print(cmp.round(3).to_string())

tex = latex_comparison_table(cmp, len(cc_w), per_month_cc.mean())
with open(os.path.join(OUT_DIR, f"table_final_comparison{SUFFIX}.tex"), "w") as fh:
    fh.write(tex)
print(f"\nLaTeX table written to table_final_comparison{SUFFIX}.tex")

## Rank transformation

In [ ]:
cc_r = rank_transform(cc_w, seed=SEED)
im_r = rank_transform(im_w, missing=missing_mask.loc[im_w.index], seed=SEED)

st = cc_r[CHARS].describe().T
print(f"Complete case after ranking: min {st['min'].min():.3f}, max {st['max'].max():.3f}, "
      f"mean of means {st['mean'].mean():.4f}, missing {int(cc_r[CHARS].isna().sum().sum())}")

dup = max(cc_r.groupby(level="date")[c].apply(lambda v: v.duplicated().sum()).sum() for c in CHARS)
print(f"Tied ranks remaining in the complete-case sample: {dup}")
mm = missing_mask.loc[im_r.index]
zero_ok = all((im_r.loc[mm[c], c] == 0).all() for c in CHARS)
print(f"Imputed sample: imputed values equal to zero in every month: {zero_ok}")

print(f"\nTarget (not ranked): mean {100*cc_r['target'].mean():.3f}% per month, "
      f"std {100*cc_r['target'].std():.2f}%")

## Country and sector indicators

In [ ]:
master = pd.read_csv(MASTER_PATH, dtype=str)
info = master.set_index("Symbol")[["country", "Sector"]]
for d in (cc_r, im_r):
    sym = d.index.get_level_values("symbol")
    d["country"] = sym.map(info["country"])
    d["sector"] = sym.map(info["Sector"])

print("Complete case by country (securities):")
print(cc_r.reset_index().groupby("country")["symbol"].nunique().to_string())
print(f"Sectors: {cc_r['sector'].nunique()}")

## Save

In [ ]:
KEEP = ["target", "retx", "MV", "country", "sector"] + CHARS
f_cc = f"estimation_complete_case{SUFFIX}.parquet"
f_im = f"estimation_imputed{SUFFIX}.parquet"
cc_r[KEEP].reset_index().to_parquet(os.path.join(OUT_DIR, f_cc), index=False)
im_r[KEEP + ["n_imputed"]].reset_index().to_parquet(os.path.join(OUT_DIR, f_im), index=False)

for f in [f_cc, f_im]:
    p = os.path.join(OUT_DIR, f)
    print(f"  {f:40s} {len(pd.read_parquet(p)):>10,} rows  ({os.path.getsize(p)/1e6:.1f} MB)")

### Tied ranks in issues_1_12

Share of observations at the modal value and number of distinct tied ranks across months (Section 3.4.6).

In [ ]:
v = cc_w["issues_1_12"]
d = v.index.get_level_values("date")
mode = v.groupby(level="date").agg(lambda s: s.value_counts().index[0])
is_mode = v.eq(mode.reindex(d).values)
r = cc_w.groupby(level="date")["issues_1_12"].rank(method="average")
n = cc_w.groupby(level="date")["issues_1_12"].transform("size")
tied = (2 * (r - 0.5) / n - 1)[is_mode].groupby(level="date").first()
print(f"{100*is_mode.mean():.0f}% at the modal value; {tied.round(10).nunique()} distinct tied ranks over {d.nunique()} months")